[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/tensor-forge/blob/main/fable_folder/notebooks/11_art_of_creation.ipynb)

# ⚒️ Act III · Quest 11 — The Art of Creation

*Networks that don't just recognize — they invent. Autoencoders, VAEs, GANs, and diffusion.*

⬅️ [10_debugging_dojo](10_debugging_dojo.ipynb)  •  [12_art_of_action](12_art_of_action.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## Flipping the question

So far: *"given this input, what is it?"* Generative modeling asks the reverse:
*"what does the data **distribution** look like — and can I draw new samples from it?"*

Four ideas, in ascending order of magic. All run in seconds on CPU.

| Model | Core trick |
|-------|-----------|
| **Autoencoder** | squeeze data through a bottleneck; whatever survives is the essence |
| **VAE** | make the bottleneck a *distribution* → sampling becomes possible |
| **GAN** | a forger and a detective train against each other |
| **Diffusion** | learn to undo noise, then sculpt samples out of pure static |

### 1 — Autoencoder: compress glyphs through a 2-D keyhole

In [ ]:
# --- The Glyph dataset: ✕ ◯ ┼ ╱  (self-contained, no torchvision needed) ----
GLYPHS = ["cross", "ring", "plus", "slash"]

def _draw_glyph(cls, size=20, rng=None):
    rng = rng or torch.Generator().manual_seed(0)
    img = torch.zeros(size, size)
    ys, xs = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    jx = int(torch.randint(-2, 3, (1,), generator=rng))   # random jitter
    jy = int(torch.randint(-2, 3, (1,), generator=rng))
    c = size // 2
    if cls == 0:    # cross ✕ : two diagonals
        img[((xs - ys).abs() + (jx - jy)).abs() <= 1] = 1.0
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    elif cls == 1:  # ring ◯
        r2 = (xs - c - jx) ** 2 + (ys - c - jy) ** 2
        img[(r2 >= 25) & (r2 <= 49)] = 1.0
    elif cls == 2:  # plus ┼
        img[(ys - c - jy).abs() <= 1] = 1.0
        img[(xs - c - jx).abs() <= 1] = 1.0
    else:           # slash ╱ : one diagonal
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    img = img + 0.08 * torch.randn(size, size, generator=rng)
    return img.clamp(0, 1)

def make_glyphs(n_per_class=300, size=20, seed=0):
    rng = torch.Generator().manual_seed(seed)
    X = torch.stack([_draw_glyph(c, size, rng) for c in range(4) for _ in range(n_per_class)])
    y = torch.arange(4).repeat_interleave(n_per_class)
    perm = torch.randperm(len(y), generator=rng)
    return X[perm].unsqueeze(1), y[perm]   # (N, 1, 20, 20), (N,)

In [ ]:
X, y = make_glyphs(n_per_class=250)
X_flat = X.flatten(1)                                   # (1000, 400)

class AutoEncoder(nn.Module):
    def __init__(self, latent=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(400, 128), nn.ReLU(), nn.Linear(128, latent))
        self.dec = nn.Sequential(nn.Linear(latent, 128), nn.ReLU(), nn.Linear(128, 400), nn.Sigmoid())
    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), z

ae = AutoEncoder().to(device)
opt = torch.optim.Adam(ae.parameters(), lr=2e-3)
Xd = X_flat.to(device)
for epoch in range(400):
    recon, z = ae(Xd)
    loss = F.mse_loss(recon, Xd)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"reconstruction MSE: {loss.item():.4f}   (400 pixels squeezed through 2 numbers!)")

In [ ]:
# Left: originals vs reconstructions. Right: THE LATENT SPACE — the 2 numbers, colored by class.
fig = plt.figure(figsize=(11, 3.5))
with torch.no_grad():
    recon, z = ae(Xd)
for i in range(6):
    axo = fig.add_subplot(2, 12, i + 1);      axo.imshow(X[i, 0], cmap="gray"); axo.axis("off")
    axr = fig.add_subplot(2, 12, i + 13);     axr.imshow(recon[i].reshape(20, 20).cpu(), cmap="gray"); axr.axis("off")
axz = fig.add_subplot(1, 2, 2)
sc = axz.scatter(z[:, 0].cpu(), z[:, 1].cpu(), c=y, cmap="tab10", s=8)
axz.legend(handles=sc.legend_elements()[0], labels=GLYPHS, fontsize=8)
axz.set_title("latent space: the AE separated the classes ON ITS OWN")
plt.suptitle("top: originals · bottom: reconstructions"); plt.show()

Nobody told the autoencoder about classes — it discovered the glyph types just by learning to
compress. **Representation learning** in one picture.

### 2 — VAE: make the keyhole sample-able

An AE's latent space has *holes* — decode a random point and you get junk. A **VAE** encodes
each input as a *Gaussian* (`μ`, `σ`) and penalizes the latents for straying from `N(0, I)`.
Result: a smooth space you can sample from.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent=2):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(400, 128), nn.ReLU())
        self.mu = nn.Linear(128, latent)
        self.logvar = nn.Linear(128, latent)
        self.dec = nn.Sequential(nn.Linear(latent, 128), nn.ReLU(), nn.Linear(128, 400), nn.Sigmoid())
    def forward(self, x):
        h = self.body(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)    # the reparameterization trick
        return self.dec(z), mu, logvar

vae = VAE().to(device)
opt = torch.optim.Adam(vae.parameters(), lr=2e-3)
for epoch in range(500):
    recon, mu, logvar = vae(Xd)
    rec = F.mse_loss(recon, Xd)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    loss = rec + 0.005 * kl
    opt.zero_grad(); loss.backward(); opt.step()
print(f"recon {rec.item():.4f} | KL {kl.item():.3f}")

# Sample brand-new glyphs from pure noise!
with torch.no_grad():
    fresh = vae.dec(torch.randn(12, 2).to(device)).reshape(-1, 20, 20).cpu()
fig, ax = plt.subplots(1, 12, figsize=(12, 1.5))
for i in range(12):
    ax[i].imshow(fresh[i], cmap="gray"); ax[i].axis("off")
plt.suptitle("glyphs that never existed — decoded from random latent points"); plt.show()

### 3 — GAN: the forger and the detective

Target: points arranged in a **heart** ♥. The **generator** G never sees the data — it only
gets feedback from the **discriminator** D, which is simultaneously learning to tell real from
fake. Two networks locked in an arms race.

In [ ]:
def sample_heart(n):
    t = torch.rand(n) * 2 * math.pi
    x = 16 * torch.sin(t) ** 3
    y = 13 * torch.cos(t) - 5 * torch.cos(2 * t) - 2 * torch.cos(3 * t) - torch.cos(4 * t)
    return (torch.stack([x, y], dim=1) / 16.0) + 0.03 * torch.randn(n, 2)

real = sample_heart(2000).to(device)

G = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2)).to(device)
D = nn.Sequential(nn.Linear(2, 64), nn.LeakyReLU(0.2), nn.Linear(64, 64), nn.LeakyReLU(0.2), nn.Linear(64, 1)).to(device)
optG = torch.optim.Adam(G.parameters(), lr=1e-3, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=1e-3, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

snapshots = {}
for step in range(4000):
    # detective: real -> 1, forged -> 0
    idx = torch.randint(0, len(real), (256,))
    fake = G(torch.randn(256, 8, device=device)).detach()
    lossD = bce(D(real[idx]), torch.ones(256, 1, device=device)) + \
            bce(D(fake), torch.zeros(256, 1, device=device))
    optD.zero_grad(); lossD.backward(); optD.step()

    # forger: make D say 1
    fake = G(torch.randn(256, 8, device=device))
    lossG = bce(D(fake), torch.ones(256, 1, device=device))
    optG.zero_grad(); lossG.backward(); optG.step()

    if step in (0, 500, 1500, 3999):
        with torch.no_grad():
            snapshots[step] = G(torch.randn(1500, 8, device=device)).cpu()

fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
for i, (s, pts) in enumerate(snapshots.items()):
    ax[i].scatter(real[:, 0].cpu(), real[:, 1].cpu(), s=3, alpha=0.15, c="gray")
    ax[i].scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.5, c="crimson")
    ax[i].set_title(f"step {s}"); ax[i].set_xlim(-1.6, 1.6); ax[i].set_ylim(-1.6, 1.2); ax[i].axis("off")
plt.suptitle("the forger learns the heart, having NEVER seen it directly ♥"); plt.show()

### 4 — Diffusion: sculpting from static

The idea behind Stable Diffusion, in miniature. **Forward**: gradually noise the data to pure
static over `T` steps. **Learn**: a network that predicts the added noise at any step.
**Generate**: start from static, subtract predicted noise step by step.

In [ ]:
T = 60
betas = torch.linspace(1e-4, 0.04, T)
alpha_bar = torch.cumprod(1 - betas, dim=0).to(device)

class NoisePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, (t.float() / T).unsqueeze(-1)], dim=-1))

eps_net = NoisePredictor().to(device)
opt = torch.optim.Adam(eps_net.parameters(), lr=1e-3)
for step in range(4000):
    x0 = sample_heart(256).to(device)
    t = torch.randint(0, T, (256,), device=device)
    noise = torch.randn_like(x0)
    ab = alpha_bar[t].unsqueeze(-1)
    xt = ab.sqrt() * x0 + (1 - ab).sqrt() * noise      # jump straight to noise level t
    loss = F.mse_loss(eps_net(xt, t), noise)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"noise-prediction loss: {loss.item():.4f}")

In [ ]:
@torch.no_grad()
def sculpt(n=1500):
    x = torch.randn(n, 2, device=device)               # pure static
    frames = [x.cpu().clone()]
    for t in reversed(range(T)):
        eps = eps_net(x, torch.full((n,), t, device=device))
        a, ab = 1 - betas[t], alpha_bar[t]
        x = (x - betas[t].to(device) / (1 - ab).sqrt() * eps) / a.sqrt().to(device)
        if t > 0:
            x = x + betas[t].sqrt().to(device) * torch.randn_like(x)
        if t in (T - 1, T // 2, T // 5, 0):
            frames.append(x.cpu().clone())
    return frames

frames = sculpt()
fig, ax = plt.subplots(1, len(frames), figsize=(3 * len(frames), 3))
for i, f in enumerate(frames):
    ax[i].scatter(f[:, 0], f[:, 1], s=3, alpha=0.5, c="purple")
    ax[i].set_xlim(-2, 2); ax[i].set_ylim(-2, 2); ax[i].axis("off")
    ax[i].set_title("static" if i == 0 else f"denoising…" if i < len(frames) - 1 else "♥")
plt.suptitle("diffusion: a heart sculpted out of pure noise"); plt.show()

Swap the 2-D points for images and the MLP for a U-Net, and this *identical algorithm* is
Stable Diffusion. You now hold the core of modern generative AI.

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda s: "bottleneck" in s.lower() or "compress" in s.lower() or "latent" in s.lower() or "squeeze" in s.lower(),
    "one word for what forces an autoencoder to learn structure")
_register("tight_ae", 15,
    lambda mse: mse < 0.02,
    "improve the AE (bigger hidden layer, latent=4, more epochs) until recon MSE < 0.02; submit loss.item()")
_register("latent_walk", 15,
    lambda imgs: torch.is_tensor(imgs) and imgs.shape == (8, 400),
    "decode 8 points interpolated between two random latent vectors: z = a*(1-t) + b*t; submit vae.dec(zs)")
_register("gan_roles", 10,
    lambda s: s.strip().lower() in ("discriminator", "detective", "critic", "d"),
    "which network provides ALL of the generator's learning signal?")

In [ ]:
check("warmup", "bottleneck")

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `tight_ae` (15 XP) — get the autoencoder MSE under `0.02`; submit the loss value.
- `latent_walk` (15 XP) — interpolate 8 steps between two random latent points and decode them; submit the `(8, 400)` tensor.
- `gan_roles` (10 XP) — one word: where does the generator's learning signal come from?

In [ ]:
# ⚔️ your attempts here...

# xp_report()